# Chronicle RPG — Web-Hosted MCP Server on Google Colab (Google Drive + OAuth 2.1 + Cloudflare Tunnel)

Run **Chronicle** as a persistent, cloud-hosted **Streamable HTTP MCP server (`/mcp`)** + **Web GM Studio GUI** directly from Google Colab.

### Pre-Configured for Your Google Drive Folder
- **Target Drive Folder:** `https://drive.google.com/drive/folders/1D7Omo7ckAkk5DN4f9SKPRGs4wX1yMqJl`
- Cell 1 automatically resolves that Drive folder ID across `MyDrive`, `Shareddrives`, and `Othercomputers` (Google Drive for Desktop synced laptop folders), and auto-detects both your Chronicle source (`Cargo.toml`) and your campaign (`my-campaign/campaign.db`).

### Architecture (Zero SQLite Corruption on Google Drive)
- **Persistent Storage:** Your campaign (`campaign.db` + compiled `docs/`) and cached Linux binary live in your Google Drive folder.
- **Safe Local Runtime + Background Checkpoint Sync:** Google Drive FUSE does not safely support SQLite WAL `-shm` mmap locks. This notebook copies your campaign to Colab's fast local SSD (`/content/campaign_live/`) at startup and runs a background **SQLite online-backup sync daemon** that mirrors clean snapshots of `campaign.db` and `docs/` back to Google Drive every 5 seconds (plus on shutdown) and mirrors `campaign.backup-*.db` snapshots both ways.
- **Built-In OAuth 2.1 + PKCE + Dynamic Client Registration:** Chronicle natively serves RFC 9728 (`/.well-known/oauth-protected-resource`), RFC 8414 (`/.well-known/oauth-authorization-server`), RFC 7591 (`/oauth/register`), `/oauth/authorize` (protected by your GM OAuth Passphrase), and `/oauth/token` — ready for **ChatGPT Web/Desktop**, **Antigravity**, and **Claude**.
- **Cloudflare Tunnel (Two Modes Supported):**
  1. **`quick` (Zero Account Needed):** Instant free `https://<random>.trycloudflare.com/mcp` URL (new URL each Colab session).
  2. **`named` (Permanent Custom URL via Cloudflare Zero Trust):** Uses your Cloudflare Tunnel token so your Colab MCP server ALWAYS comes back on the **exact same permanent URL** (e.g. `https://chronicle.staronshadowcompass.com/mcp`).

### Cloudflare Named Tunnel Checklist (Verify These 5 Settings in Cloudflare Zero Trust)

1. Go to [Cloudflare Zero Trust Dashboard](https://one.dash.cloudflare.com/) → **Networks → Tunnels → your tunnel → Public Hostname**.
2. Check your Public Hostname (`chronicle.staronshadowcompass.com`):
   - **Service Type:** **`HTTP`** *(NOT `HTTPS` — Chronicle speaks plain HTTP locally and Cloudflare terminates HTTPS at the edge!)*
   - **URL:** **`127.0.0.1:7878`**
3. **Do NOT put Cloudflare Access / Zero Trust Login in front of `/mcp` or `/.well-known/*`** — ChatGPT/Antigravity need direct access to Chronicle's built-in OAuth 2.1 discovery endpoints (Chronicle already protects `/mcp` and `/api/*` with OAuth 2.1 + PKCE!).
4. **Store secrets safely in Colab Secrets** (left sidebar → **Key icon 🔑**), instead of pasting tokens into notebook cells:
   - `CLOUDFLARE_TUNNEL_TOKEN` = your Cloudflare Tunnel token (`eyJhIjoi...`)
   - `CHRONICLE_OAUTH_SECRET` = your fixed GM OAuth passphrase (optional; auto-generated if omitted)
5. Keep `CLOUDFLARED_PROTOCOL = 'http2'` (default below) — Google Colab containers often drop UDP/QUIC traffic, whereas HTTP/2 over TCP/443 connects reliably.

In [1]:
#@title 1. Mount Google Drive, Resolve Folder ID & Configure OAuth / Tunnel
import glob, os, re, secrets
from google.colab import drive

drive.mount('/content/drive')

def _get_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(name)
        return val.strip() if val else ''
    except Exception:
        return ''

def _resolve_drive_folder_id(url_or_id: str, fallback_path: str) -> str:
    """Resolves a Google Drive folder URL or ID to its mounted /content/drive/... path."""
    raw = url_or_id.strip()
    if raw.startswith('/content/drive/'):
        return raw
    if fallback_path and os.path.isdir(fallback_path) and os.path.exists(os.path.join(fallback_path, 'Cargo.toml')):
        return fallback_path
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', raw)
    folder_id = m.group(1) if m else raw
    if folder_id:
        try:
            from google.colab import auth
            from googleapiclient.discovery import build
            auth.authenticate_user()
            svc = build('drive', 'v3')
            parts = []
            cur = folder_id
            for _ in range(15):
                meta = svc.files().get(
                    fileId=cur,
                    fields='id, name, parents',
                    supportsAllDrives=True,
                ).execute()
                parents = meta.get('parents') or []
                if not parents:
                    break
                parts.append(meta['name'])
                cur = parents[0]
            parts.reverse()
            candidates = [
                os.path.join('/content/drive/MyDrive', *parts),
                os.path.join('/content/drive/Shareddrives', *parts),
            ]
            for comp in sorted(glob.glob('/content/drive/Othercomputers/*')):
                candidates.append(os.path.join(comp, *parts))
            for cand in candidates:
                if os.path.isdir(cand):
                    print(f'Resolved Drive folder ID {folder_id} -> {cand}')
                    return cand
            print(f'Note: Drive API resolved {parts} but not found under mount; using fallback...')
        except Exception as exc:
            print(f'Drive API path resolution skipped ({exc}); using fallback path.')
    return fallback_path

#@markdown ### Google Drive Folder & Campaign Settings
DRIVE_FOLDER_URL_OR_ID = 'https://drive.google.com/drive/folders/1D7Omo7ckAkk5DN4f9SKPRGs4wX1yMqJl?usp=drive_link' #@param {type:"string"}
#@markdown Manual fallback path (e.g., Google Drive for Desktop synced laptop folder):
DRIVE_FALLBACK_PATH = '/content/drive/Othercomputers/My Laptop/rpg' #@param {type:"string"}
CAMPAIGN_NAME = 'my-campaign' #@param {type:"string"}
PORT = 7878 #@param {type:"integer"}

#@markdown ### Authentication Settings (OAuth 2.1 + optional Static Bearer Token)
#@markdown Leave blank to pull `CHRONICLE_OAUTH_SECRET` from Colab Secrets or auto-generate a strong passphrase:
OAUTH_SECRET = '' #@param {type:"string"}
#@markdown Optional separate Static Bearer token (leave BLANK for pure OAuth 2.1 — do NOT paste your Cloudflare Tunnel Token here!):
STATIC_BEARER_TOKEN = '' #@param {type:"string"}

#@markdown ### Cloudflare Tunnel Settings
#@markdown Choose `named` (permanent custom domain) OR `quick` (free instant `trycloudflare.com` URL):
TUNNEL_MODE = 'quick' #@param ["named", "quick"]
#@markdown `http2` is strongly recommended on Google Colab (UDP/QUIC is often dropped in Colab containers):
CLOUDFLARED_PROTOCOL = 'http2' #@param ["http2", "auto", "quic"]
#@markdown Only needed when `TUNNEL_MODE == 'named'` (leave blank to load `CLOUDFLARE_TUNNEL_TOKEN` safely from Colab Secrets):
CLOUDFLARE_TUNNEL_TOKEN = '' #@param {type:"string"}
CUSTOM_DOMAIN = '' #@param {type:"string"}

if not OAUTH_SECRET.strip():
    OAUTH_SECRET = _get_colab_secret('CHRONICLE_OAUTH_SECRET') or ('gm-' + secrets.token_urlsafe(18))
if not STATIC_BEARER_TOKEN.strip():
    STATIC_BEARER_TOKEN = _get_colab_secret('CHRONICLE_MCP_TOKEN')
if TUNNEL_MODE == 'named' and not CLOUDFLARE_TUNNEL_TOKEN.strip():
    CLOUDFLARE_TUNNEL_TOKEN = _get_colab_secret('CLOUDFLARE_TUNNEL_TOKEN')

if STATIC_BEARER_TOKEN.strip() and STATIC_BEARER_TOKEN.strip() == CLOUDFLARE_TUNNEL_TOKEN.strip():
    print('WARNING: STATIC_BEARER_TOKEN matched CLOUDFLARE_TUNNEL_TOKEN — ignoring STATIC_BEARER_TOKEN for safety!')
    STATIC_BEARER_TOKEN = ''

DRIVE_ROOT = _resolve_drive_folder_id(DRIVE_FOLDER_URL_OR_ID, DRIVE_FALLBACK_PATH)
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Auto-detect Chronicle source directory
SOURCE_DIR = None
for cand in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'source'), os.path.join(DRIVE_ROOT, 'rpg')]:
    if os.path.exists(os.path.join(cand, 'Cargo.toml')):
        SOURCE_DIR = cand
        break
if SOURCE_DIR is None:
    SOURCE_DIR = DRIVE_ROOT

# Auto-detect campaign directory
DRIVE_CAMPAIGN_DIR = None
for cand in [
    os.path.join(DRIVE_ROOT, CAMPAIGN_NAME),
    os.path.join(DRIVE_ROOT, 'campaigns', CAMPAIGN_NAME),
    os.path.join(DRIVE_ROOT, 'rpg', CAMPAIGN_NAME),
]:
    if os.path.exists(os.path.join(cand, 'campaign.db')):
        DRIVE_CAMPAIGN_DIR = cand
        break
if DRIVE_CAMPAIGN_DIR is None:
    DRIVE_CAMPAIGN_DIR = os.path.join(DRIVE_ROOT, CAMPAIGN_NAME)

DRIVE_BIN_DIR = os.path.join(DRIVE_ROOT, 'bin')
LOCAL_CAMPAIGN_DIR = f'/content/campaign_live/{CAMPAIGN_NAME}'
LOCAL_BIN = '/content/chronicle'

for d in [DRIVE_BIN_DIR, DRIVE_CAMPAIGN_DIR, os.path.dirname(LOCAL_CAMPAIGN_DIR)]:
    os.makedirs(d, exist_ok=True)

print('Drive Root        :', DRIVE_ROOT)
print('Detected Source   :', SOURCE_DIR, '(Cargo.toml found:', os.path.exists(os.path.join(SOURCE_DIR, 'Cargo.toml')), ')')
print('Drive Campaign Dir:', DRIVE_CAMPAIGN_DIR, '(campaign.db found:', os.path.exists(os.path.join(DRIVE_CAMPAIGN_DIR, 'campaign.db')), ')')
print('Local Live Dir    :', LOCAL_CAMPAIGN_DIR)
print('Tunnel Mode       :', TUNNEL_MODE, '| Protocol:', CLOUDFLARED_PROTOCOL)
print('Tunnel Token Set  :', bool(CLOUDFLARE_TUNNEL_TOKEN.strip()))
print('GM OAuth Secret   :', OAUTH_SECRET)

Mounted at /content/drive
Drive Root        : /content/drive/Othercomputers/My Laptop/rpg
Detected Source   : /content/drive/Othercomputers/My Laptop/rpg (Cargo.toml found: True )
Drive Campaign Dir: /content/drive/Othercomputers/My Laptop/rpg/my-campaign (campaign.db found: True )
Local Live Dir    : /content/campaign_live/my-campaign
Tunnel Mode       : quick | Protocol: http2
Tunnel Token Set  : False
GM OAuth Secret   : gm-UUbzgS_Ao0BXC9KbVuiltd_c


In [5]:
#@title 2. Install / Build Chronicle Linux Binary (Auto-Rebuilds When Source Files Are Newer)
import glob, os, shutil, subprocess

cached_bin = os.path.join(DRIVE_BIN_DIR, 'chronicle-linux-x86_64')
FORCE_REBUILD = False #@param {type:"boolean"}

def _latest_source_mtime(src_root: str) -> float:
    latest = 0.0
    for pattern in ['Cargo.toml', 'Cargo.lock', 'src/*.rs', 'src/*.html']:
        for p in glob.glob(os.path.join(src_root, pattern)):
            try:
                latest = max(latest, os.path.getmtime(p))
            except OSError:
                pass
    return latest

src_mtime = _latest_source_mtime(SOURCE_DIR)
bin_mtime = os.path.getmtime(cached_bin) if os.path.exists(cached_bin) else 0.0
needs_rebuild = FORCE_REBUILD or (not os.path.exists(cached_bin)) or (src_mtime > bin_mtime + 2.0)

if not needs_rebuild:
    print('Using up-to-date cached Chronicle binary from Google Drive (fast path)...')
    shutil.copy2(cached_bin, LOCAL_BIN)
    os.chmod(LOCAL_BIN, 0o755)
else:
    reason = 'FORCE_REBUILD=True' if FORCE_REBUILD else ('no cached binary' if not os.path.exists(cached_bin) else 'newer source files detected in Drive')
    print(f'Building Chronicle from source ({reason})...')
    if not os.path.exists(os.path.join(SOURCE_DIR, 'Cargo.toml')):
        raise FileNotFoundError(
            f'Cargo.toml not found in {SOURCE_DIR}. Make sure Cargo.toml, Cargo.lock, and src/ are synced to Drive.'
        )
    if shutil.which('cargo') is None and not os.path.exists('/root/.cargo/bin/cargo'):
        subprocess.run('curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y', shell=True, check=True)
    env = os.environ.copy()
    env['PATH'] = '/root/.cargo/bin:' + env.get('PATH', '')
    build_dir = '/content/chronicle-src'
    if os.path.exists(build_dir):
        shutil.rmtree(build_dir)
    shutil.copytree(
        SOURCE_DIR,
        build_dir,
        ignore=shutil.ignore_patterns('target', 'my-campaign', 'campaigns', 'bin', '.git'),
    )
    subprocess.run(['cargo', 'build', '--release'], cwd=build_dir, env=env, check=True)
    built = os.path.join(build_dir, 'target', 'release', 'chronicle')
    shutil.copy2(built, LOCAL_BIN)
    shutil.copy2(built, cached_bin)
    os.chmod(LOCAL_BIN, 0o755)
    print('Built and cached updated binary to:', cached_bin)

subprocess.run([LOCAL_BIN, '--version'], check=True)

Building Chronicle from source (newer source files detected in Drive)...
Built and cached updated binary to: /content/drive/Othercomputers/My Laptop/rpg/bin/chronicle-linux-x86_64


CompletedProcess(args=['/content/chronicle', '--version'], returncode=0)

In [ ]:
#@title 3. Sync Campaign from Google Drive to Local SSD & Start Background Drive Checkpoint Sync
import glob, os, shutil, sqlite3, subprocess, threading

if os.path.exists(LOCAL_CAMPAIGN_DIR):
    shutil.rmtree(LOCAL_CAMPAIGN_DIR)
os.makedirs(LOCAL_CAMPAIGN_DIR, exist_ok=True)

drive_db = os.path.join(DRIVE_CAMPAIGN_DIR, 'campaign.db')
local_db = os.path.join(LOCAL_CAMPAIGN_DIR, 'campaign.db')

if os.path.exists(drive_db):
    print('Restoring existing campaign from Google Drive:', drive_db)
    shutil.copy2(drive_db, local_db)
    # Also restore uncheckpointed WAL/SHM sidecars if present in Drive so no committed events are lost!
    wal_restored = False
    for suffix in ('-wal', '-shm'):
        src_sidecar = drive_db + suffix
        if os.path.exists(src_sidecar):
            shutil.copy2(src_sidecar, local_db + suffix)
            wal_restored = True
    if wal_restored:
        print('  Detected uncheckpointed SQLite WAL/SHM sidecars in Drive — merging into local DB...')
        print('  TIP: Avoid running Chronicle on Windows and Colab simultaneously to prevent Drive sync conflicts.')
        conn = sqlite3.connect(local_db, timeout=15.0)
        try:
            conn.execute('PRAGMA wal_checkpoint(TRUNCATE);')
        finally:
            conn.close()
    # Mirror existing backup snapshots from Drive -> local SSD so backup_campaign(list) & rollback see them
    restored_backups = 0
    for bpath in sorted(glob.glob(os.path.join(DRIVE_CAMPAIGN_DIR, 'campaign.backup-*.db'))):
        shutil.copy2(bpath, os.path.join(LOCAL_CAMPAIGN_DIR, os.path.basename(bpath)))
        restored_backups += 1
    # Print restored event count for instant DB identity verification
    try:
        conn = sqlite3.connect(f'file:{local_db}?mode=ro', uri=True)
        ev_count, max_seq = conn.execute('SELECT COUNT(*), COALESCE(MAX(seq), 0) FROM events').fetchone()
        conn.close()
        print(f'  Restored DB verified: {ev_count} events (latest seq #{max_seq}), {restored_backups} backup(s).')
    except Exception as exc:
        print(f'  (Could not inspect event count: {exc})')
    drive_att = os.path.join(DRIVE_CAMPAIGN_DIR, 'docs', 'attachments', 'files')
    if os.path.exists(drive_att):
        local_att = os.path.join(LOCAL_CAMPAIGN_DIR, 'docs', 'attachments', 'files')
        os.makedirs(os.path.dirname(local_att), exist_ok=True)
        shutil.copytree(drive_att, local_att, dirs_exist_ok=True)
    subprocess.run([LOCAL_BIN, 'compile', '-c', LOCAL_CAMPAIGN_DIR], check=True)
else:
    print('No campaign.db found in Drive yet — initializing fresh empty campaign...')
    subprocess.run([
        LOCAL_BIN, 'init', LOCAL_CAMPAIGN_DIR,
        '--name', CAMPAIGN_NAME,
        '--description', 'Cloud-hosted Chronicle RPG campaign',
        '--ruleset', 'System-agnostic RPG core'
    ], check=True)

def sync_live_to_drive():
    """Uses SQLite online backup API so Google Drive always receives a clean, corruption-free snapshot + backups."""
    if not os.path.exists(local_db):
        return
    tmp_snap = '/content/campaign.snap.db'
    src = sqlite3.connect(local_db, timeout=10.0)
    dst = sqlite3.connect(tmp_snap)
    try:
        src.backup(dst)
    finally:
        dst.close()
        src.close()
    shutil.copy2(tmp_snap, drive_db + '.tmp')
    os.replace(drive_db + '.tmp', drive_db)
    # Remove any stale Drive -wal/-shm sidecars now that Drive holds a complete checkpointed single-file DB
    for suffix in ('-wal', '-shm'):
        stale = drive_db + suffix
        if os.path.exists(stale):
            try:
                os.remove(stale)
            except OSError:
                pass
    # Mirror any newly created local backups back to Drive
    for bpath in glob.glob(os.path.join(LOCAL_CAMPAIGN_DIR, 'campaign.backup-*.db')):
        dst_b = os.path.join(DRIVE_CAMPAIGN_DIR, os.path.basename(bpath))
        if not os.path.exists(dst_b):
            shutil.copy2(bpath, dst_b)
    local_docs = os.path.join(LOCAL_CAMPAIGN_DIR, 'docs')
    drive_docs = os.path.join(DRIVE_CAMPAIGN_DIR, 'docs')
    if os.path.exists(local_docs):
        shutil.copytree(local_docs, drive_docs, dirs_exist_ok=True)

SYNC_STOP = threading.Event()
_last_state = (0.0, 0)

def _sync_loop():
    global _last_state
    while not SYNC_STOP.is_set():
        try:
            mt = max(
                os.path.getmtime(local_db) if os.path.exists(local_db) else 0.0,
                os.path.getmtime(local_db + '-wal') if os.path.exists(local_db + '-wal') else 0.0,
            )
            bcount = len(glob.glob(os.path.join(LOCAL_CAMPAIGN_DIR, 'campaign.backup-*.db')))
            cur_state = (mt, bcount)
            if cur_state != _last_state:
                sync_live_to_drive()
                _last_state = cur_state
        except Exception as exc:
            print(f'[drive-sync] warning: {exc}')
        SYNC_STOP.wait(5.0)

sync_live_to_drive()
SYNC_THREAD = threading.Thread(target=_sync_loop, daemon=True)
SYNC_THREAD.start()
print('Campaign ready and Google Drive checkpoint sync active (every 5s, with WAL merge & backup mirroring).')

In [ ]:
#@title 4. Start Chronicle OAuth 2.1 MCP Server + Cloudflare Tunnel (With Live Logs & Public Self-Check)
import collections, json, re, shutil, subprocess, threading, time, urllib.error, urllib.request

# Install cloudflared Linux binary if missing
if shutil.which('cloudflared') is None:
    print('Downloading cloudflared...')
    subprocess.run(
        'curl -L --silent https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared',
        shell=True, check=True
    )
subprocess.run(['cloudflared', '--version'], check=True)

# Validate tunnel mode config before launching
if TUNNEL_MODE == 'named':
    if not CLOUDFLARE_TUNNEL_TOKEN.strip():
        raise ValueError('TUNNEL_MODE is "named" but CLOUDFLARE_TUNNEL_TOKEN is empty. Add it to Colab Secrets (CLOUDFLARE_TUNNEL_TOKEN) or Cell 1.')
    public_url = CUSTOM_DOMAIN.strip().rstrip('/')
    if not public_url.startswith('https://'):
        raise ValueError('CUSTOM_DOMAIN must start with https:// (e.g. https://chronicle.staronshadowcompass.com)')
else:
    public_url = None

# Start Cloudflare Tunnel and ALWAYS drain its stdout/stderr in a background thread
# (prevents 64KB pipe deadlock AND surfaces any tunnel/origin errors immediately!)
cf_logs = collections.deque(maxlen=40)
cf_connected = threading.Event()
cf_quick_url = [None]

def _drain_cloudflared(proc):
    for raw_line in iter(proc.stdout.readline, ''):
        line = raw_line.rstrip()
        if not line:
            continue
        cf_logs.append(line)
        m = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
        if m and cf_quick_url[0] is None:
            cf_quick_url[0] = m.group(1)
        if 'Registered tunnel connection' in line:
            cf_connected.set()
            print(f'[cloudflared] {line}', flush=True)
        elif any(k in line for k in ('ERR ', 'WRN ', 'error', 'failed', 'Unable to reach', 'refused')):
            print(f'[cloudflared] {line}', flush=True)

if TUNNEL_MODE == 'named':
    print(f'Starting Cloudflare Named Tunnel ({CLOUDFLARED_PROTOCOL}) for {public_url} ...')
    cf_cmd = ['cloudflared', 'tunnel', '--no-autoupdate', '--protocol', CLOUDFLARED_PROTOCOL, 'run', '--token', CLOUDFLARE_TUNNEL_TOKEN.strip()]
else:
    print(f'Starting Cloudflare Quick Tunnel ({CLOUDFLARED_PROTOCOL})...')
    cf_cmd = ['cloudflared', 'tunnel', '--no-autoupdate', '--protocol', CLOUDFLARED_PROTOCOL, '--url', f'http://127.0.0.1:{PORT}']

cf_proc = subprocess.Popen(cf_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
cf_thread = threading.Thread(target=_drain_cloudflared, args=(cf_proc,), daemon=True)
cf_thread.start()

if TUNNEL_MODE == 'quick':
    deadline = time.time() + 30
    while time.time() < deadline and cf_quick_url[0] is None:
        if cf_proc.poll() is not None:
            raise RuntimeError('cloudflared exited early:\n' + '\n'.join(cf_logs))
        time.sleep(0.25)
    public_url = cf_quick_url[0]
    if not public_url:
        cf_proc.terminate()
        raise RuntimeError('Failed to obtain Quick Tunnel URL within 30s:\n' + '\n'.join(cf_logs))
else:
    # Wait up to 15s for at least one tunnel edge registration
    cf_connected.wait(timeout=15)
    if cf_proc.poll() is not None:
        raise RuntimeError('cloudflared named tunnel exited early:\n' + '\n'.join(cf_logs))

# Start Chronicle Streamable HTTP MCP + Web GM Studio with OAuth 2.1 enabled
cmd = [
    LOCAL_BIN, 'mcp',
    '--campaign', LOCAL_CAMPAIGN_DIR,
    '--transport', 'http',
    '--host', '127.0.0.1',
    '--port', str(PORT),
    '--oauth',
    '--oauth-secret', OAUTH_SECRET,
    '--public-url', public_url,
]
if STATIC_BEARER_TOKEN.strip():
    cmd += ['--bearer-token', STATIC_BEARER_TOKEN.strip()]

srv_proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# Wait for local /health
local_ok = False
for _ in range(30):
    if srv_proc.poll() is not None:
        raise RuntimeError('Chronicle server exited early:\n' + (srv_proc.stdout.read() if srv_proc.stdout else ''))
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=2) as r:
            if r.status == 200:
                local_ok = True
                break
    except Exception:
        time.sleep(0.5)

if not local_ok:
    raise RuntimeError('Chronicle failed local /health check on 127.0.0.1:' + str(PORT))

# End-to-end PUBLIC tunnel verification (checks DNS + TLS + Cloudflare Tunnel -> Chronicle)
print(f'Verifying public Cloudflare Tunnel -> {public_url}/health ...', flush=True)
public_ok = False
last_err = None
for attempt in range(1, 9):
    try:
        req = urllib.request.Request(f'{public_url}/health', headers={'User-Agent': 'Chronicle-SelfCheck/1.0'})
        with urllib.request.urlopen(req, timeout=5) as r:
            body = r.read().decode('utf-8', errors='replace')
            if r.status == 200 and 'ok' in body:
                # Also verify OAuth 2.1 Protected Resource Metadata is reachable publicly
                req2 = urllib.request.Request(f'{public_url}/.well-known/oauth-protected-resource', headers={'User-Agent': 'Chronicle-SelfCheck/1.0'})
                with urllib.request.urlopen(req2, timeout=5) as r2:
                    meta = json.loads(r2.read().decode('utf-8'))
                    assert meta.get('resource') == f'{public_url}/mcp'
                public_ok = True
                break
    except urllib.error.HTTPError as e:
        last_err = f'HTTP {e.code} {e.reason}: {e.read().decode("utf-8", errors="replace")[:200]}'
    except Exception as e:
        last_err = str(e)
    time.sleep(2.5)

print('=' * 74)
if public_ok:
    print('PUBLIC CLOUDFLARE TUNNEL VERIFIED END-TO-END! (Health + OAuth 2.1 OK)')
else:
    print('WARNING: Local Chronicle is UP, but PUBLIC tunnel check failed!')
    print(f'  Last error: {last_err}')
    print('  Check Cloudflare Zero Trust -> Tunnels -> Public Hostname:')
    print(f'    - Hostname    : {public_url.replace("https://", "")}')
    print(f'    - Service Type: HTTP  (NOT HTTPS!)')
    print(f'    - Service URL : 127.0.0.1:{PORT}')
    print('    - Make sure Cloudflare Access / Bot Fight Mode is NOT blocking /mcp or /.well-known/*')
    print('  Recent cloudflared logs:')
    for l in list(cf_logs)[-10:]:
        print('   ', l)
print('=' * 74)
print(f'Tunnel Mode         : {TUNNEL_MODE} ({CLOUDFLARED_PROTOCOL})')
print(f'MCP Server URL      : {public_url}/mcp')
print(f'Web GM Studio GUI   : {public_url}/')
print(f'OAuth Discovery     : {public_url}/.well-known/oauth-protected-resource')
print(f'GM OAuth Passphrase : {OAUTH_SECRET}')
if STATIC_BEARER_TOKEN.strip():
    print(f'Static Bearer Token : {STATIC_BEARER_TOKEN.strip()}')
print('-' * 74)
print('ChatGPT (Settings -> Apps / Connectors -> Developer Mode):')
print(f'  URL: {public_url}/mcp  |  Auth: OAuth -> enter the GM OAuth Passphrase above')
print('Antigravity (~/.gemini/config/mcp_config.json):')
print(f'  "chronicle-colab": {{ "url": "{public_url}/mcp" }}')
print('=' * 74)
print('Live request logs below. Keep this cell running while playing; stop cleanly for final Drive sync.')

try:
    while srv_proc.poll() is None:
        line = srv_proc.stdout.readline()
        if line:
            print(line, end='', flush=True)
except KeyboardInterrupt:
    print('\nShutting down cleanly and performing final Google Drive sync...')
finally:
    srv_proc.terminate()
    cf_proc.terminate()
    SYNC_STOP.set()
    sync_live_to_drive()
    print('Final Google Drive sync complete!')

Starting Cloudflare Quick Tunnel (http2)...
[cloudflared] 2026/11/22 10:33:49 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.
[cloudflared] 2026-11-22T10:33:49Z INF Registered tunnel connection connIndex=0 connection=2d82fac2-3a0b-4991-ac16-79dd28e70690 event=0 ip=198.41.192.7 location=ord14 protocol=http2
Verifying public Cloudflare Tunnel -> https://end-romance-perceived-worked.trycloudflare.com/health ...
  Last error: <urlopen error [Errno -2] Name or service not known>
  Check Cloudflare Zero Trust -> Tunnels -> Public Hostname:
    - Hostname    : end-romance-perceived-worked.trycloudflare.com
    - Service Type: HTTP  (NOT HTTPS!)
    - Service URL : 127.0.0.1:7878
    - Make sure Cloudflare Access / Bot Fight Mode is NOT blocking /mcp or /.well-known/*
  Recent cloudflared logs:
    2026-11-22T10:33:55Z INF |  SUMMARY: Environment is healthy. cloudflared